# Cuaderno A: prueba V4 en ABIDES (semillas 1000 y 2000)

Prueba de la recompensa del Ejecutor **contra el precio de llegada** (variante V4: lo que cuesta cada compra frente al precio al que empezó la orden), con **penalización de 30 bps por cada acción sin comprar y γ = 1**. La confirmación anterior (misma penalización y γ, recompensa original) evitó que la red se quedara esperando pero no bajó el costo (+3,0 bps sobre el control); aquí se prueba si ver el costo de llenarse tarde lo arregla. Ver `docs/confirmacion_abides_penalizacion_resultado_PS.md`.

Las semillas de red son las mismas de la confirmación anterior, así que se comparan con ellas y con el control.

Ejecuta las celdas **en orden** (o "Ejecutar todas"; si la celda 1 corta la ejecución, vuelve a lanzar "Ejecutar todas"). Dura unos **120 minutos**. Todo se guarda en Drive en la carpeta **`slippage_colab/resultados/v4_pen30_g1_A`** (carpeta propia, para que los dos cuadernos no se mezclen) y al final se arma `v4_pen30_g1_A.zip`.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama con la recompensa contra el precio de llegada; debe mostrar un commit y un 2)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-abides-recompensa-is-llegada
!cd /content/slippage && git log --oneline -1 && grep -c "is_llegada" scripts/colab/entrenamiento_inicial_ps.py

In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Entrenamientos (2 semillas, ≈ 50 min cada una) y control

In [ ]:
# Entrenamiento de 1.000 episodios, recompensa contra el precio de llegada, penalización de 30 bps y gamma = 1 (semilla de red 1000, evaluada con 40 semillas; ≈ 50 min)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/entrenamiento_inicial_ps.py --recompensa is_llegada --gamma 1.0 --penalizacion-bps 30 --seed 1000 --episodes 1000 --eval-seeds 40 --out-dir {RESULTS_DIR}/v4_pen30_g1_A/d3_opcion0_s1000 --max-minutes 150 2>&1 | grep -v UserWarning

In [ ]:
# Entrenamiento de 1.000 episodios, recompensa contra el precio de llegada, penalización de 30 bps y gamma = 1 (semilla de red 2000, evaluada con 40 semillas; ≈ 50 min)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/entrenamiento_inicial_ps.py --recompensa is_llegada --gamma 1.0 --penalizacion-bps 30 --seed 2000 --episodes 1000 --eval-seeds 40 --out-dir {RESULTS_DIR}/v4_pen30_g1_A/d3_opcion0_s2000 --max-minutes 150 2>&1 | grep -v UserWarning

## Control: las mismas redes sin entrenar, muestreando la política (40 semillas de evaluación)

In [ ]:
# Control: red sin entrenar muestreando, en las mismas 40 semillas de evaluación (no entrena; unos minutos)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/control_muestreado_ps.py --semillas 1000 2000 --eval-seeds 40 --out-dir {RESULTS_DIR}/v4_pen30_g1_A 2>&1 | grep -v UserWarning

## Zip final

In [ ]:
# CELDA FINAL: revisa lo que se guardó y arma un zip pequeño para descargar (sin checkpoints)
import os, zipfile
base = f'{RESULTS_DIR}/v4_pen30_g1_A'
archivos = []
for carpeta, _, fs in os.walk(base):
    if 'checkpoints_' in carpeta: continue
    for a in fs:
        if a.endswith(('.json', '.csv')):
            archivos.append(os.path.join(carpeta, a))
print(len(archivos), "archivos en", base)
for a in sorted(archivos): print(" ", os.path.relpath(a, base))
zpath = f'{RESULTS_DIR}/v4_pen30_g1_A.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print("zip listo:", zpath, os.path.getsize(zpath), "bytes  -> descárgalo desde slippage_colab/resultados en Drive")